# Fast KMeans++ on MNIST

This tutorial compares FastKMeans++ with scikit-learn's KMeans on MNIST. It checks adjusted Rand index for `K=10`, then compares runtime for `K=10` and `K=100`.

## Load MNIST

The data is scaled to `[0, 1]` before clustering. A fixed seed keeps centroid initialization reproducible.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.datasets import fetch_openml
from sklearn.metrics import adjusted_rand_score

from fastkmeanspp import KMeans

# Fix the seed so both implementations use the same reproducible setup.
random_state = 42
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = np.asarray(mnist.data, dtype=np.float32) / 255
y = np.asarray(mnist.target, dtype=np.int8)
X.shape, y.shape

((70000, 784), (70000,))

## Compare adjusted Rand index for `K=10`

ARI measures agreement with the known digit labels while correcting for chance. Higher values indicate better clustering alignment.

In [ ]:
def fit_model(model, X):
    start = time.perf_counter()
    model.fit(X)
    return model, time.perf_counter() - start


# Compare ARI over 10 runs between the two implementations.
ari_10 = []
for seed in range(10):
    sklearn_10 = SklearnKMeans(
        n_clusters=10,
        n_init=1,
        max_iter=20,
        random_state=seed,
    )
    fast_10 = KMeans(
        n_clusters=10,
        n_iter=20,
        random_state=seed,
    )

    sklearn_10.fit(X)
    fast_10.fit(X)

    ari_10.append(
        {
            "scikit-learn": adjusted_rand_score(y, sklearn_10.labels_),
            "fastkmeanspp": adjusted_rand_score(y, fast_10.labels_),
        }
    )

ari_10 = pd.DataFrame(ari_10)
ari_summary = pd.DataFrame(
    {
        "ARI mean": ari_10.mean(),
        "ARI std": ari_10.std(),
    }
)
ari_summary.style.format("{:.3f}")

,ARI mean,ARI std
scikit-learn,0.343,0.031
fastkmeanspp,0.359,0.031


## Compare speed for `K=10`

The first FastKMeans++ fit warms up the native distance kernel. The reported run measures a fresh fit after that warmup.

In [ ]:
# Time the two implementations for K=10.
_, sklearn_time_10 = fit_model(
    SklearnKMeans(
        n_clusters=10,
        n_init=1,
        max_iter=20,
        random_state=random_state,
    ),
    X,
)
_, fast_time_10 = fit_model(
    KMeans(
        n_clusters=10,
        n_iter=20,
        random_state=random_state,
    ),
    X,
)

speed_10 = pd.DataFrame(
    {
        "fit time (s)": [
            sklearn_time_10,
            fast_time_10,
        ]
    },
    index=[
        "scikit-learn",
        "fastkmeanspp",
    ],
)
speed_10.style.format("{:.3f}")

,fit time (s)
scikit-learn,3.398
fastkmeanspp,0.784


## Compare speed for `K=100`

These timings measure one fit of each implementation on the same MNIST matrix. Repeat the cells for a more stable machine-specific benchmark.

In [ ]:
# Time the two implementations for K=100.
_, sklearn_time_100 = fit_model(
    SklearnKMeans(
        n_clusters=100,
        n_init=1,
        max_iter=20,
        random_state=random_state,
    ),
    X,
)
_, fast_time_100 = fit_model(
    KMeans(
        n_clusters=100,
        n_iter=20,
        random_state=random_state,
    ),
    X,
)

speed_100 = pd.DataFrame(
    {"fit time (s)": [sklearn_time_100, fast_time_100]},
    index=["scikit-learn", "fastkmeanspp"],
)
speed_100.style.format("{:.3f}")

,fit time (s)
scikit-learn,32.245
fastkmeanspp,1.876
